# Description:
This notebook will be used to evaluate the performance of the pix2pix/cyclegan models by quantifying the quality of the generated images. 
We have done this before using the L2 Loss or MSE Loss, SSIM , PSNR 
Lets do it again but with lot of clarity and using the reproducible scripts to avoid the large fucking footprint of the codebase ~ 
~ SENSE OF URGENCY ~ 

"Thats where it flipped the switch in me where i was like "okay, fuck you, watch this" "
Carmy ~ Bear Season 1 Episode 6

We need to load the folders containing both real and fake images like target and style transferred images


In [1]:
import os
from skimage.metrics import structural_similarity as ssim
from skimage.metrics import peak_signal_noise_ratio as psnr
import numpy as np
import pandas as pd 
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import numpy as np
import os
import cv2





def compute_mse_error_map(fake_image, real_image):
    return np.sqrt(np.square(fake_image - real_image))
def compute_ssim_error_map(fake_image, real_image):
    return ssim(fake_image, real_image, data_range=np.max(fake_image) - np.min(fake_image), multichannel=False)
def compute_psnr_error_map(fake_image, real_image):
    return psnr(fake_image, real_image, data_range=np.max(fake_image) - np.min(fake_image))
def normalize_image(image):
    image = image.astype(np.float32)
    image = image / 255.0
    return image

In [2]:
# high res images
images_folder = "/home/user/haris/pytorch-CycleGAN-and-pix2pix/results/pix2pix_source2target_highres/pix2pix_source2target_highres/test_latest/images"
# Low res images
# images_folder = "/home/user/haris/pytorch-CycleGAN-and-pix2pix/results/pix2pix_source2target/test_latest/images"

# interested_images = [x for x in os.listdir(images_folder) if x.contains("_fake_B.png", "_real_B.png")]
fake_images_paths = [os.path.join(images_folder, x) for x in sorted(os.listdir(images_folder)) if x.endswith("_fake_B.png")]
real_images_paths = [os.path.join(images_folder, x) for x in sorted(os.listdir(images_folder)) if x.endswith("_real_B.png")]


In [3]:
def get_error_map(error_map,title,path,preview=True,save=True):
    """
    shows and saves the error map with the title in specified path
    """
    error_map_new = np.zeros_like(error_map)
    error_map_new[error_map > error_map.mean()] = 1
    fig = plt.figure(figsize=(10, 5))
    plt.imshow(error_map_new,cmap="jet")
    plt.colorbar()
    plt.axis("off")
    plt.tight_layout()
    if preview:
        plt.show()
    else:
        plt.close(fig)
    if save:
        # Saved figure is white means not saved properly
        fig.savefig(path,bbox_inches="tight",pad_inches=0)
        plt.close(fig)

dict_df = {"sample_id":[],"ssim":[],"psnr":[],"mse":[],"image_path":[]}
for sample_id in range(len(fake_images_paths)):
    fake_image = cv2.imread(fake_images_paths[sample_id],cv2.IMREAD_GRAYSCALE)
    real_image = cv2.imread(real_images_paths[sample_id],cv2.IMREAD_GRAYSCALE)
    fake_image = normalize_image(fake_image).astype(np.float32)
    real_image = normalize_image(real_image).astype(np.float32)
    ssim_value = compute_ssim_error_map(fake_image, real_image)
    psnr_value = compute_psnr_error_map(fake_image, real_image)
    mse_value = compute_mse_error_map(fake_image, real_image)
    # visualize the error map
    title = f"MSE: {mse_value.mean():.2f}, SSIM: {ssim_value:.2f}, PSNR: {psnr_value:.2f}"
    sample_id_str = fake_images_paths[sample_id].split("/")[-1].split(".")[0].split("_")[0]
    path = f"{images_folder}/{sample_id_str}_mse.png"
    get_error_map(error_map=mse_value,title=title,path=path,preview=False,save=True)
    # print(f"SSIM: {ssim_value}, PSNR: {psnr_value}")
    dict_df["sample_id"].append(sample_id_str)
    dict_df["ssim"].append(round(ssim_value,3))
    dict_df["psnr"].append(round(psnr_value,3))
    dict_df["mse"].append(round(mse_value.mean(),3))
    dict_df["image_path"].append(fake_images_paths[sample_id])
    # break
df = pd.DataFrame(dict_df)
# df.to_csv(f"{images_folder}/error_analysis.csv",index=False)

In [5]:
images_folder

'/home/user/haris/pytorch-CycleGAN-and-pix2pix/results/pix2pix_source2target_highres/pix2pix_source2target_highres/test_latest/images'

In [6]:
df.to_csv("/".join(images_folder.split("/")[:-1])+"/error_analysis.csv",index=False)

# Analysis of Error Dataframes

In [5]:
import pandas as pd
df_low_res_errors = pd.read_csv("/home/user/haris/pytorch-CycleGAN-and-pix2pix/results/pix2pix_source2target/test_latest/error_analysis.csv")
df_high_res_errors = pd.read_csv("/home/user/haris/pytorch-CycleGAN-and-pix2pix/results/pix2pix_source2target_highres/pix2pix_source2target_highres/test_latest/error_analysis.csv")

display(df_low_res_errors.describe().round(4))

display(df_high_res_errors.describe().round(4))





,ssim,psnr,mse
count,647.0000,647.0000,647.0000
mean,0.2134,16.1078,0.1133
std,0.0601,1.6284,0.0260
min,0.0900,10.0840,0.0550
25%,0.1680,15.1320,0.0960
50%,0.2040,16.1560,0.1110
75%,0.2470,17.2225,0.1260
max,0.4350,20.3870,0.2450


,ssim,psnr,mse
count,647.0000,647.0000,647.0000
mean,0.2174,16.1355,0.1126
std,0.0661,1.8032,0.0295
min,0.0800,9.3070,0.0460
25%,0.1670,15.1235,0.0930
50%,0.2080,16.2770,0.1090
75%,0.2585,17.3515,0.1270
max,0.4810,21.9210,0.2930
